In [37]:
from physics.simulation import mcfm, msq
from physics.hzz import zz2l2v
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
    #     "\\\\usepackage{fontspec}"
    #     "\\\\setmainfont{Computer Modern Roman}"  # or Computer Modern if installed
})


In [38]:
COMPONENT = msq.Component.SBI

with open('data/zz2l2v/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : np.prod(xsec['gg_sbi']),
    msq.Component.SIG : np.prod(xsec['gg_sig']),
    msq.Component.INT : np.prod(xsec['gg_int']),
    msq.Component.BKG : np.prod(xsec['gg_bkg'])
}

filenames = {
    msq.Component.SBI : 'data/zz2l2v/ggZZ_sbi/events_*.csv',
    msq.Component.SIG : 'data/zz2l2v/ggZZ_sig/events_*.csv',
    msq.Component.INT : 'data/zz2l2v/ggZZ_int/events_*.csv',
    msq.Component.BKG : 'data/zz2l2v/ggZZ_bkg/events_*.csv'
}

In [39]:
events = mcfm.from_csv(cross_section=xsec[COMPONENT], file_path=filenames[COMPONENT])
print(len(events.weights))
events = zz2l2v.analyze(events)

1861242
Inclusive |  16.43810621100001
MET > 100 GeV |  1.8073575060965836
DPhillMET > 2.5 |  1.8073575060965836
DRll < 1.8 |  1.6187288355783769


In [40]:
xobs = events.kinematics['zz_mt'].to_numpy()
nxbins = 31
xmin, xmax = 270, 1200.0
# xbins = np.logspace(np.log10(xmin), np.log10(xmax), nxbins + 1)
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [41]:
color_sm = 'black'
line_sm = '--'

In [42]:
from physics.variables import c6_vals, cH_vals
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

mod_c6 = c6.Modifier(baseline=COMPONENT, events=events, c6_points=[-20,-10,0,10,20])
wt_c6, prob_c6 = mod_c6.modify(c6_vals)

[[ 1.00000000e+00 -2.87712066e-04  4.07710037e-05 -6.61710240e-08
   1.24376755e-08]
 [ 1.00000000e+00 -8.53581317e-04 -1.07926279e-04 -8.37872103e-07
   9.17838537e-08]
 [ 1.00000000e+00 -2.61916892e-03 -4.34914051e-05 -1.18033643e-06
   1.00237680e-07]
 ...
 [ 1.00000000e+00 -2.37646323e-03  2.43805430e-04 -1.87320626e-06
   8.88419760e-08]
 [ 1.00000000e+00 -9.87133631e-03  3.82660678e-04 -5.76198556e-06
   2.36044718e-07]
 [ 1.00000000e+00 -2.18940680e-03  6.56622004e-05 -2.80619500e-06
   1.24673353e-07]]


In [43]:
h_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sm.fill(xobs, weight=events.weights)

h_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs, weight=wt_c6[:,i_c6])
    h_c6.append(h)

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [2, 1]}, figsize=(6,6), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6]))
l_sm = ax1.stairs(h_sm.values(), xbins, color=color_sm, linestyle=line_sm)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\\sigma / \\mathrm{d} m_{\\mathrm{T}}^{ZZ}\\ \\mathrm{[fb/GeV]}$', fontsize=15)

for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$C_H = {sgn}{abs(cH_vals[i_c6]):d}$')
l_sm.set_label('$\\mathrm{SM}$')
ax1.legend(frameon=False, fontsize=12)

ax2.stairs(h_sm.values() / h_sm.values(), h_sm.axes[0].edges, color=color_sm, linestyle=line_sm)
for i_c6, c6_val in enumerate(c6_vals):
    ax2.stairs(h_c6[i_c6].values() / h_sm.values(), h_sm.axes[0].edges, color=c6_colors[i_c6], linestyle=c6_lines[i_c6])

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.95,1.25)

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{\\mathrm{T}}^{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False)

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('gg2l2v_bsm.pdf')
plt.close()